In [1]:
import os 
import numpy as np 
import pandas as pd 
import requests 
import sys
from dotenv import load_dotenv
import json

In [2]:
data_path = "data"


In [3]:
def get_github_token():
    load_dotenv()
    token = os.getenv("GITHUB_TOKEN")
    if token is None:
        print("GITHUB_TOKEN not found in environment variables.")
        sys.exit(1)
    return token

In [4]:
def create_github_link(repository_url, issue_number):
    return f"{repository_url}/issues/{issue_number}"

In [5]:
def fetch_issue_contents(repository_url, issue_number, token):
    """
    Fetches comprehensive issue data including:
    - Basic issue details
    - All comments
    - All events (labels, assignments, etc.)
    - Timeline (unified view of comments and events)
    """
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
        "User-Agent": "simple-github-issues-script",
    }
    
    issue_data = {}
    
    # 1. Fetch basic issue details
    issue_url = create_github_link(repository_url=repository_url, issue_number=issue_number)
    response = requests.get(issue_url, headers=headers)
    if response.status_code != 200:
        print(f"Failed to fetch issue: {response.status_code}")
        return {}
    issue_data['issue'] = response.json()
    
    # 2. Fetch all comments
    comments_url = f"{issue_url}/comments"
    comments = []
    page = 1
    while True:
        response = requests.get(comments_url, headers=headers, params={"page": page, "per_page": 100})
        if response.status_code != 200:
            print(f"Failed to fetch comments: {response.status_code}")
            break
        page_comments = response.json()
        if not page_comments:
            break
        comments.extend(page_comments)
        page += 1
    issue_data['comments'] = comments
    
    # 3. Fetch all events (labels, assignments, closes, reopens, etc.)
    events_url = f"{issue_url}/events"
    events = []
    page = 1
    while True:
        response = requests.get(events_url, headers=headers, params={"page": page, "per_page": 100})
        if response.status_code != 200:
            print(f"Failed to fetch events: {response.status_code}")
            break
        page_events = response.json()
        if not page_events:
            break
        events.extend(page_events)
        page += 1
    issue_data['events'] = events
    
    # 4. Fetch timeline (unified chronological view)
    # Note: Requires special accept header
    timeline_url = f"{issue_url}/timeline"
    timeline_headers = headers.copy()
    timeline_headers["Accept"] = "application/vnd.github.mockingbird-preview+json"
    timeline = []
    page = 1
    while True:
        response = requests.get(timeline_url, headers=timeline_headers, params={"page": page, "per_page": 100})
        if response.status_code != 200:
            print(f"Failed to fetch timeline: {response.status_code}")
            break
        page_timeline = response.json()
        if not page_timeline:
            break
        timeline.extend(page_timeline)
        page += 1
    issue_data['timeline'] = timeline
    
    return issue_data


In [6]:
def extract_comment_info(issue_content):
    """
    Extracts relevant information from issue comments.
    Returns a list of dictionaries with the desired fields for each comment.
    """
    comments_info = []
    for comment in issue_content['comments']:
        comments_info.append({
            'commenter': comment['user']['login'],
            'comment_text': comment['body'].strip(),
            'commenter_association': comment['author_association']
        })  
    return comments_info

In [7]:
def extract_issue_info(issue_content):
    """
    Extracts relevant information from fetched issue content.
    Returns a dictionary with the desired fields.
    """
    comment_info = extract_comment_info(issue_content)
    
    return {
        'html_url': issue_content['issue']['html_url'],
        'issue_reporter': issue_content['issue']['user']['login'],
        'issue_reporter_association': issue_content['issue']['author_association'],
        'comment_count': len(issue_content['comments']),
        'commenters': [c['commenter'] for c in comment_info],
        'comment_texts': [c['comment_text'] for c in comment_info],
        'commenter_associations': [c['commenter_association'] for c in comment_info]
    }

### Fetch and Update All Issues
Now we can fetch all issues and update the DataFrame with the new columns.

In [17]:
# Load objective_issues.csv
objective_df = pd.read_csv(os.path.join(data_path, "objective_issues.csv"))
print(f"Total issues in objective_issues.csv: {len(objective_df)}")
print(f"Columns: {objective_df.columns.tolist()}")

# Sample 1000 random issues
random_seed = 42
sample_size = min(1000, len(objective_df))  # In case there are fewer than 1000 issues
random_sample = objective_df.sample(n=sample_size, random_state=random_seed)
print(f"\nSampled {len(random_sample)} random issues")
print(random_sample.head())

# Save sampled issues to CSV
sampled_csv_path = os.path.join(data_path, "sampled_issues_1000.csv")
random_sample.to_csv(sampled_csv_path, index=False)
print(f"\nSaved sampled issues to: {sampled_csv_path}")

Total issues in objective_issues.csv: 817743
Columns: ['repository_url', 'issue_number', 'title_processed', 'body_processed', 'label', 'label_cat', 'test_tag']

Sampled 1000 random issues
                                           repository_url  issue_number  \
282078       https://api.github.com/repos/mcMMO-Dev/mcMMO          3191   
200570  https://api.github.com/repos/spring-projects/s...         23574   
730077  https://api.github.com/repos/otros-systems/otr...           471   
793920            https://api.github.com/repos/usgs/swarm           249   
356486    https://api.github.com/repos/Shyam101/calculate             1   

                                          title_processed  \
282078                        cannot repair certain items   
200570  move code snippet reference documentation actu...   
730077     importing log4j xml fails contains binary data   
793920                    include picks sac header export   
356486                                     improve butto

In [18]:
# Load the sampled issues
sampled_df = pd.read_csv(os.path.join(data_path, "sampled_issues_1000.csv"))
print(f"Loaded {len(sampled_df)} sampled issues")

# Initialize new columns with appropriate data types
sampled_df['html_url'] = None
sampled_df['html_url'] = sampled_df['html_url'].astype('object')
sampled_df['issue_reporter'] = None
sampled_df['issue_reporter'] = sampled_df['issue_reporter'].astype('object')
sampled_df['issue_reporter_association'] = None
sampled_df['issue_reporter_association'] = sampled_df['issue_reporter_association'].astype('object')
sampled_df['comment_count'] = None
sampled_df['comment_count'] = sampled_df['comment_count'].astype('Int64')
sampled_df['commenters'] = [[] for _ in range(len(sampled_df))]
sampled_df['comment_texts'] = [[] for _ in range(len(sampled_df))]
sampled_df['commenter_associations'] = [[] for _ in range(len(sampled_df))]

# Fetch and enhance all 1000 issues
token = get_github_token()
failed_issues = []

for idx, row in sampled_df.iterrows():
    print(f"Fetching issue {idx + 1}/{len(sampled_df)}: {row['repository_url']} #{row['issue_number']}")
    
    try:
        issue_content = fetch_issue_contents(
            repository_url=row['repository_url'],
            issue_number=row['issue_number'],
            token=token
        )
        
        if issue_content:  # Check if fetch was successful
            issue_info = extract_issue_info(issue_content)
            for key, value in issue_info.items():
                sampled_df.at[idx, key] = value
            print(f"  ✓ Success: {issue_info['comment_count']} comments")
        else:
            failed_issues.append((idx, row['repository_url'], row['issue_number']))
            print(f"  ✗ Failed to fetch")
    except Exception as e:
        failed_issues.append((idx, row['repository_url'], row['issue_number']))
        print(f"  ✗ Error: {str(e)}")

print(f"\n\n{'='*60}")
print(f"Enhancement Complete!")
print(f"Successfully enhanced: {len(sampled_df) - len(failed_issues)}/{len(sampled_df)}")
print(f"Failed: {len(failed_issues)}")
if failed_issues:
    print(f"\nFailed issues:")
    for idx, repo, issue_num in failed_issues[:10]:  # Show first 10 failures
        print(f"  - Index {idx}: {repo} #{issue_num}")
    if len(failed_issues) > 10:
        print(f"  ... and {len(failed_issues) - 10} more")

Loaded 1000 sampled issues
Fetching issue 1/1000: https://api.github.com/repos/mcMMO-Dev/mcMMO #3191
  ✓ Success: 2 comments
Fetching issue 2/1000: https://api.github.com/repos/spring-projects/spring-boot #23574
  ✓ Success: 2 comments
Fetching issue 2/1000: https://api.github.com/repos/spring-projects/spring-boot #23574
  ✓ Success: 1 comments
Fetching issue 3/1000: https://api.github.com/repos/otros-systems/otroslogviewer #471
  ✓ Success: 1 comments
Fetching issue 3/1000: https://api.github.com/repos/otros-systems/otroslogviewer #471
  ✓ Success: 1 comments
Fetching issue 4/1000: https://api.github.com/repos/usgs/swarm #249
  ✓ Success: 1 comments
Fetching issue 4/1000: https://api.github.com/repos/usgs/swarm #249
  ✓ Success: 3 comments
Fetching issue 5/1000: https://api.github.com/repos/Shyam101/calculate #1
  ✓ Success: 3 comments
Fetching issue 5/1000: https://api.github.com/repos/Shyam101/calculate #1
  ✓ Success: 0 comments
Fetching issue 6/1000: https://api.github.com/repos/v

In [19]:
# Save the enhanced DataFrame
enhanced_csv_path = os.path.join(data_path, "sampled_issues_1000_enhanced.csv")
sampled_df.to_csv(enhanced_csv_path, index=False)
print(f"Saved enhanced dataset to: {enhanced_csv_path}")

# Display summary statistics
print("\nSummary Statistics:")
print(f"Total issues: {len(sampled_df)}")
print(f"Issues with data: {sampled_df['html_url'].notna().sum()}")
print(f"\nComment count statistics:")
print(sampled_df['comment_count'].describe())
print(f"\nIssue reporter associations:")
print(sampled_df['issue_reporter_association'].value_counts())

Saved enhanced dataset to: data\sampled_issues_1000_enhanced.csv

Summary Statistics:
Total issues: 1000
Issues with data: 476

Comment count statistics:
count       476.0
mean     2.485294
std      4.511248
min           0.0
25%           0.0
50%           1.0
75%           3.0
max          48.0
Name: comment_count, dtype: Float64

Issue reporter associations:
issue_reporter_association
NONE            144
CONTRIBUTOR     117
MEMBER           94
OWNER            83
COLLABORATOR     38
Name: count, dtype: int64


In [13]:
df = pd.read_csv(os.path.join(data_path, "sampled_issues_1000_enhanced.csv"))

## Dataset Summary Analysis
Comprehensive analysis of the enhanced dataset for presentation

In [14]:
# Basic Dataset Information
print("="*70)
print("DATASET OVERVIEW")
print("="*70)
print(f"Total number of issues: {len(df)}")
print(f"Total number of columns: {len(df.columns)}")
print(f"\nColumn names:")
for col in df.columns:
    print(f"  - {col}")
print(f"\nDataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

DATASET OVERVIEW
Total number of issues: 1000
Total number of columns: 14

Column names:
  - repository_url
  - issue_number
  - title_processed
  - body_processed
  - label
  - label_cat
  - test_tag
  - html_url
  - issue_reporter
  - issue_reporter_association
  - comment_count
  - commenters
  - comment_texts
  - commenter_associations

Dataset shape: (1000, 14)
Memory usage: 1.69 MB


In [24]:
# Labels per label_cat and test_tag
print("\n" + "="*70)
print("DISTRIBUTION BY LABEL CATEGORY AND TEST TAG")
print("="*70)

if 'label_cat' in df.columns:
    # Overall label_cat distribution
    print("\nLabel Category Distribution:")
    label_cat_counts = df['label_cat'].value_counts()
    for label_cat, count in label_cat_counts.items():
        percentage = (count / len(df)) * 100
        print(f"  {label_cat}: {count} ({percentage:.1f}%)")
    
    if 'test_tag' in df.columns:
        # Cross-tabulation
        crosstab = pd.crosstab(df['label_cat'], df['test_tag'], margins=True)
        print("\n\nCross-tabulation (label_cat x test_tag):")
        print("Note: test_tag appears to be 0 (train) or 1 (test)")
        print(crosstab)
        
        print("\n\nDetailed breakdown:")
        print("-" * 70)
        for label_cat in sorted(df['label_cat'].dropna().unique()):
            print(f"\n{label_cat}:")
            test_counts = df[df['label_cat'] == label_cat]['test_tag'].value_counts()
            for test_tag, count in test_counts.items():
                percentage = (count / len(df[df['label_cat'] == label_cat])) * 100
                tag_name = "test" if test_tag == 1 else "train"
                print(f"  {tag_name} (test_tag={test_tag}): {count} ({percentage:.1f}%)")
            print(f"  Total: {len(df[df['label_cat'] == label_cat])}")
        
        # Overall test_tag distribution
        print("\n" + "-" * 70)
        print("\nOverall Test Tag Distribution:")
        test_counts = df['test_tag'].value_counts()
        for test_tag, count in test_counts.items():
            percentage = (count / len(df)) * 100
            tag_name = "test" if test_tag == 1 else "train"
            print(f"  {tag_name} (test_tag={test_tag}): {count} ({percentage:.1f}%)")

# Original label distribution
if 'label' in df.columns:
    print("\n" + "-" * 70)
    print("\nOriginal Label Distribution (top 15):")
    label_counts = df['label'].value_counts().head(15)
    for label, count in label_counts.items():
        percentage = (count / len(df)) * 100
        print(f"  {label}: {count} ({percentage:.1f}%)")


DISTRIBUTION BY LABEL CATEGORY AND TEST TAG

Label Category Distribution:
  bug: 461 (46.1%)
  feature: 427 (42.7%)
  support: 112 (11.2%)


Cross-tabulation (label_cat x test_tag):
Note: test_tag appears to be 0 (train) or 1 (test)
test_tag     0    1   All
label_cat                
bug        370   91   461
feature    340   87   427
support     90   22   112
All        800  200  1000


Detailed breakdown:
----------------------------------------------------------------------

bug:
  train (test_tag=0): 370 (80.3%)
  test (test_tag=1): 91 (19.7%)
  Total: 461

feature:
  train (test_tag=0): 340 (79.6%)
  test (test_tag=1): 87 (20.4%)
  Total: 427

support:
  train (test_tag=0): 90 (80.4%)
  test (test_tag=1): 22 (19.6%)
  Total: 112

----------------------------------------------------------------------

Overall Test Tag Distribution:
  train (test_tag=0): 800 (80.0%)
  test (test_tag=1): 200 (20.0%)

----------------------------------------------------------------------

Original La

In [16]:
# Comment Analysis
print("\n" + "="*70)
print("COMMENT ANALYSIS")
print("="*70)

if 'comment_count' in df.columns:
    print("\nComment Count Statistics:")
    print(df['comment_count'].describe())
    
    print("\n\nComment Distribution:")
    print(f"  Issues with 0 comments: {(df['comment_count'] == 0).sum()} ({(df['comment_count'] == 0).sum() / len(df) * 100:.1f}%)")
    print(f"  Issues with 1-5 comments: {((df['comment_count'] >= 1) & (df['comment_count'] <= 5)).sum()} ({((df['comment_count'] >= 1) & (df['comment_count'] <= 5)).sum() / len(df) * 100:.1f}%)")
    print(f"  Issues with 6-10 comments: {((df['comment_count'] >= 6) & (df['comment_count'] <= 10)).sum()} ({((df['comment_count'] >= 6) & (df['comment_count'] <= 10)).sum() / len(df) * 100:.1f}%)")
    print(f"  Issues with 11-20 comments: {((df['comment_count'] >= 11) & (df['comment_count'] <= 20)).sum()} ({((df['comment_count'] >= 11) & (df['comment_count'] <= 20)).sum() / len(df) * 100:.1f}%)")
    print(f"  Issues with 20+ comments: {(df['comment_count'] > 20).sum()} ({(df['comment_count'] > 20).sum() / len(df) * 100:.1f}%)")
    
    print(f"\n  Maximum comments on a single issue: {df['comment_count'].max()}")
    print(f"  Minimum comments: {df['comment_count'].min()}")
    print(f"  Average comments per issue: {df['comment_count'].mean():.2f}")
    print(f"  Median comments per issue: {df['comment_count'].median():.1f}")
    print(f"  Total comments across all issues: {df['comment_count'].sum()}")
else:
    print("Column 'comment_count' not found in the dataset")


COMMENT ANALYSIS

Comment Count Statistics:
count    476.000000
mean       2.485294
std        4.511248
min        0.000000
25%        0.000000
50%        1.000000
75%        3.000000
max       48.000000
Name: comment_count, dtype: float64


Comment Distribution:
  Issues with 0 comments: 145 (14.5%)
  Issues with 1-5 comments: 279 (27.9%)
  Issues with 6-10 comments: 30 (3.0%)
  Issues with 11-20 comments: 18 (1.8%)
  Issues with 20+ comments: 4 (0.4%)

  Maximum comments on a single issue: 48.0
  Minimum comments: 0.0
  Average comments per issue: 2.49
  Median comments per issue: 1.0
  Total comments across all issues: 1183.0


In [17]:
# Issue Reporter Analysis
print("\n" + "="*70)
print("ISSUE REPORTER ANALYSIS")
print("="*70)

if 'issue_reporter_association' in df.columns:
    print("\nIssue Reporter Association Distribution:")
    reporter_assoc = df['issue_reporter_association'].value_counts()
    for assoc, count in reporter_assoc.items():
        percentage = (count / len(df)) * 100
        print(f"  {assoc}: {count} ({percentage:.1f}%)")
    
    print(f"\n  Total unique issue reporters: {df['issue_reporter'].nunique() if 'issue_reporter' in df.columns else 'N/A'}")
else:
    print("Column 'issue_reporter_association' not found in the dataset")


ISSUE REPORTER ANALYSIS

Issue Reporter Association Distribution:
  NONE: 144 (14.4%)
  CONTRIBUTOR: 117 (11.7%)
  MEMBER: 94 (9.4%)
  OWNER: 83 (8.3%)
  COLLABORATOR: 38 (3.8%)

  Total unique issue reporters: 450


In [18]:
# Repository Analysis
print("\n" + "="*70)
print("REPOSITORY ANALYSIS")
print("="*70)

if 'repository_url' in df.columns:
    print("\nNumber of unique repositories:")
    print(f"  {df['repository_url'].nunique()} repositories")
    
    print("\n\nTop 10 repositories by issue count:")
    repo_counts = df['repository_url'].value_counts().head(10)
    for i, (repo, count) in enumerate(repo_counts.items(), 1):
        percentage = (count / len(df)) * 100
        print(f"  {i}. {repo}: {count} issues ({percentage:.1f}%)")
else:
    print("Column 'repository_url' not found in the dataset")


REPOSITORY ANALYSIS

Number of unique repositories:
  825 repositories


Top 10 repositories by issue count:
  1. https://api.github.com/repos/vaadin/framework: 6 issues (0.6%)
  2. https://api.github.com/repos/dbeaver/dbeaver: 6 issues (0.6%)
  3. https://api.github.com/repos/Graylog2/graylog2-server: 6 issues (0.6%)
  4. https://api.github.com/repos/molgenis/molgenis: 6 issues (0.6%)
  5. https://api.github.com/repos/orientechnologies/orientdb: 5 issues (0.5%)
  6. https://api.github.com/repos/ContainX/openstack4j: 5 issues (0.5%)
  7. https://api.github.com/repos/zaproxy/zaproxy: 5 issues (0.5%)
  8. https://api.github.com/repos/phonegap/phonegap-plugin-push: 5 issues (0.5%)
  9. https://api.github.com/repos/ReactiveX/RxJava: 4 issues (0.4%)
  10. https://api.github.com/repos/dita-ot/dita-ot: 4 issues (0.4%)


In [23]:
# Comments by Test Tag and Label Category
print("\n" + "="*70)
print("COMMENTS BY TEST TAG AND LABEL CATEGORY")
print("="*70)

if 'comment_count' in df.columns:
    
    if 'label_cat' in df.columns:
        print("\n\nAverage comments by label category:")
        avg_by_label = df.groupby('label_cat')['comment_count'].agg(['mean', 'median', 'std', 'min', 'max', 'count'])
        print(avg_by_label)
        
else:
    print("Required columns not found for this analysis")


COMMENTS BY TEST TAG AND LABEL CATEGORY


Average comments by label category:
               mean  median       std  min   max  count
label_cat                                              
bug        2.688372     1.0  4.952164  0.0  39.0    215
feature    1.779343     1.0  2.542682  0.0  19.0    213
support    4.708333     3.0  7.573634  0.0  48.0     48


In [20]:
# Data Quality Check
print("\n" + "="*70)
print("DATA QUALITY CHECK")
print("="*70)

print("\nMissing values per column:")
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
}).sort_values('Missing Count', ascending=False)
print(missing_df[missing_df['Missing Count'] > 0])

if len(missing_df[missing_df['Missing Count'] > 0]) == 0:
    print("  No missing values found!")

print(f"\n\nData completeness: {((1 - df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100):.2f}%")


DATA QUALITY CHECK

Missing values per column:
                            Missing Count  Percentage
issue_reporter                        524        52.4
html_url                              524        52.4
comment_count                         524        52.4
issue_reporter_association            524        52.4


Data completeness: 85.03%


In [21]:
# Key Insights Summary
print("\n" + "="*70)
print("KEY INSIGHTS SUMMARY FOR PRESENTATION")
print("="*70)

print("\n📊 DATASET SIZE:")
print(f"   • Total Issues: {len(df)}")
print(f"   • Total Columns: {len(df.columns)}")

if 'label_cat' in df.columns:
    print("\n🏷️  LABEL CATEGORIES:")
    for label_cat, count in df['label_cat'].value_counts().items():
        print(f"   • {label_cat}: {count} ({count/len(df)*100:.1f}%)")

if 'test_tag' in df.columns:
    print("\n🔖 TRAIN/TEST SPLIT:")
    for test_tag, count in df['test_tag'].value_counts().items():
        tag_name = "test" if test_tag == 1 else "train"
        print(f"   • {tag_name}: {count} ({count/len(df)*100:.1f}%)")

if 'comment_count' in df.columns:
    print("\n💬 COMMENTS:")
    print(f"   • Total Comments: {int(df['comment_count'].sum())}")
    print(f"   • Average per Issue: {df['comment_count'].mean():.2f}")
    print(f"   • Median: {df['comment_count'].median():.0f}")
    print(f"   • Max: {int(df['comment_count'].max())}")
    print(f"   • Issues with 0 comments: {(df['comment_count'] == 0).sum()} ({(df['comment_count'] == 0).sum()/len(df)*100:.1f}%)")

if 'repository_url' in df.columns:
    print("\n📦 REPOSITORIES:")
    print(f"   • Unique Repositories: {df['repository_url'].nunique()}")
    
if 'issue_reporter_association' in df.columns:
    print("\n👥 ISSUE REPORTERS:")
    top_assoc = df['issue_reporter_association'].value_counts().head(3)
    for assoc, count in top_assoc.items():
        print(f"   • {assoc}: {count} ({count/len(df)*100:.1f}%)")

if 'label' in df.columns:
    print("\n🏆 TOP 5 ORIGINAL LABELS:")
    top_labels = df['label'].value_counts().head(5)
    for label, count in top_labels.items():
        print(f"   • {label}: {count} ({count/len(df)*100:.1f}%)")

print("\n" + "="*70)


KEY INSIGHTS SUMMARY FOR PRESENTATION

📊 DATASET SIZE:
   • Total Issues: 1000
   • Total Columns: 14

🏷️  LABEL CATEGORIES:
   • bug: 461 (46.1%)
   • feature: 427 (42.7%)
   • support: 112 (11.2%)

💬 COMMENTS:
   • Total Comments: 1183.0
   • Average per Issue: 2.49
   • Median: 1
   • Max: 48.0
   • Issues with 0 comments: 145 (14.5%)

📦 REPOSITORIES:
   • Unique Repositories: 825

👥 ISSUE REPORTERS:
   • NONE: 144 (14.4%)
   • CONTRIBUTOR: 117 (11.7%)
   • MEMBER: 94 (9.4%)



In [22]:
# Quick check of columns and sample data
print("Column dtypes:")
print(df.dtypes)
print("\n\nFirst few rows:")
print(df.head(3))
print("\n\nChecking 'label' column unique values:")
print(df['label'].value_counts() if 'label' in df.columns else "Label column not found")

Column dtypes:
repository_url                 object
issue_number                    int64
title_processed                object
body_processed                 object
label                          object
label_cat                      object
test_tag                        int64
html_url                       object
issue_reporter                 object
issue_reporter_association     object
comment_count                 float64
commenters                     object
comment_texts                  object
commenter_associations         object
dtype: object


First few rows:
                                      repository_url  issue_number  \
0       https://api.github.com/repos/mcMMO-Dev/mcMMO          3191   
1  https://api.github.com/repos/spring-projects/s...         23574   
2  https://api.github.com/repos/otros-systems/otr...           471   

                                     title_processed  \
0                        cannot repair certain items   
1  move code snippet referen

In [25]:
# Distribution of Labels per Label Category
print("\n" + "="*70)
print("DISTRIBUTION OF LABELS PER LABEL CATEGORY")
print("="*70)

if 'label' in df.columns and 'label_cat' in df.columns:
    # For each label_cat, show the distribution of original labels
    for label_cat in sorted(df['label_cat'].unique()):
        print(f"\n{label_cat.upper()}:")
        print("-" * 70)
        
        # Get all labels for this category
        labels_in_cat = df[df['label_cat'] == label_cat]['label'].value_counts()
        total_in_cat = len(df[df['label_cat'] == label_cat])
        
        for label, count in labels_in_cat.items():
            percentage = (count / total_in_cat) * 100
            print(f"  {label:30s}: {count:4d} ({percentage:5.1f}%)")
        
        print(f"\n  Total issues in {label_cat}: {total_in_cat}")
else:
    print("Columns 'label' and/or 'label_cat' not found in the dataset")


DISTRIBUTION OF LABELS PER LABEL CATEGORY

BUG:
----------------------------------------------------------------------
  bug                           :  449 ( 97.4%)
  defect                        :   12 (  2.6%)

  Total issues in bug: 461

FEATURE:
----------------------------------------------------------------------
  enhancement                   :  341 ( 79.9%)
  feature                       :   50 ( 11.7%)
  feature request               :   24 (  5.6%)
  improvement                   :   12 (  2.8%)

  Total issues in feature: 427

SUPPORT:
----------------------------------------------------------------------
  question                      :   65 ( 58.0%)
  documentation                 :   17 ( 15.2%)
  help wanted                   :    9 (  8.0%)
  docs                          :    7 (  6.2%)
  support                       :    4 (  3.6%)
  type: documentation           :    3 (  2.7%)
  type: question                :    3 (  2.7%)
  more info needed              : 